# Python Fundamentals - Finance Examples

This mini notebook contains examples related to financial workflows.

## Course flow
  - A: Example 1 (Private Equity)
  - B: Example 2 (Audit)
  - C: Example 3 (Treasury)


In [1]:
import csv

# Relative path for local/cloud portability
DATA_DIR = "data"


def section(title):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)


def read_csv(filename):
    with open(f"{DATA_DIR}/{filename}", newline="") as file:
        reader = csv.DictReader(file)
        return list(reader)


# Keep this helper for consistency with the full fundamentals notebook.
def median(values):
    values = sorted(values)
    n = len(values)
    middle = n // 2
    if n % 2 == 1:
        return values[middle]
    return (values[middle - 1] + values[middle]) / 2

## Example 1) Private Equity - Deal Screening

Compute core deal metrics and apply a basic investment screen:
- EBITDA Margin
- EV / EBITDA
- Net Debt / EBITDA

In [ ]:
section("1) Private Equity - Deal Screening")

rows = read_csv("pe_targets.csv")

for row in rows:
    company = row["Company"]
    revenue = float(row["Revenue"])
    ebitda = float(row["EBITDA"])
    ev = float(row["EV"])
    debt = float(row["Debt"])
    cash = float(row["Cash"])

    ebitda_margin = ebitda / revenue
    ev_to_ebitda = ev / ebitda
    net_debt_to_ebitda = (debt - cash) / ebitda

    is_target = (
        ev_to_ebitda < 10
        and ebitda_margin > 0.12
        and net_debt_to_ebitda < 4
    )

    print(f"{company}")
    print(f"  EBITDA Margin:      {ebitda_margin:.4f}")
    print(f"  EV / EBITDA:        {ev_to_ebitda:.2f}")
    print(f"  Net Debt / EBITDA:  {net_debt_to_ebitda:.2f}")

    if is_target:
        print("  POSSIBLE TARGET")


4) Private Equity - Deal Screening
Alpha Co
  EBITDA Margin:      0.1273
  EV / EBITDA:        8.93
  Net Debt / EBITDA:  2.86
  POSSIBLE TARGET
Beta LLC
  EBITDA Margin:      0.1222
  EV / EBITDA:        9.55
  Net Debt / EBITDA:  2.95
  POSSIBLE TARGET
Gamma Inc
  EBITDA Margin:      0.1895
  EV / EBITDA:        13.33
  Net Debt / EBITDA:  1.00
Delta Group
  EBITDA Margin:      0.1167
  EV / EBITDA:        8.29
  Net Debt / EBITDA:  2.91
Echo Ltd
  EBITDA Margin:      0.1429
  EV / EBITDA:        8.75
  Net Debt / EBITDA:  2.65
  POSSIBLE TARGET
Falcon Corp
  EBITDA Margin:      0.1192
  EV / EBITDA:        8.87
  Net Debt / EBITDA:  2.77


## Example 2) CPA / Audit - Journal Entry Balance Check

Use dictionary aggregation by `Entry ID` to verify whether total debits equal total credits.

In [ ]:
section("2) CPA / Audit - Journal Entry Balance Check")

rows = read_csv("journal_entries.csv")
entries = {}

for row in rows:
    entry_id = row["Entry ID"]
    debit = float(row["Debit"])
    credit = float(row["Credit"])

    if entry_id not in entries:
        entries[entry_id] = {"Debit": 0.0, "Credit": 0.0}

    entries[entry_id]["Debit"] += debit
    entries[entry_id]["Credit"] += credit

for entry_id, totals in entries.items():
    debit_total = totals["Debit"]
    credit_total = totals["Credit"]
    balanced = debit_total == credit_total

    print(
        f"{entry_id} | Debit: {debit_total:,.2f} | "
        f"Credit: {credit_total:,.2f} | Balanced: {balanced}"
    )
    if not balanced:
        print("  FLAG UNBALANCED ENTRY")


5) CPA / Audit - Journal Entry Balance Check
JE001 | Debit: 15,000.00 | Credit: 15,000.00 | Balanced: True
JE002 | Debit: 8,000.00 | Credit: 8,000.00 | Balanced: True
JE003 | Debit: 25,000.00 | Credit: 25,000.00 | Balanced: True
JE004 | Debit: 4,200.00 | Credit: 3,900.00 | Balanced: False
  FLAG UNBALANCED ENTRY
JE005 | Debit: 11,000.00 | Credit: 11,000.00 | Balanced: True


## Example 3) Treasury - Daily Cash Forecast

Track daily liquidity with a running ending-cash balance and flag low-cash days.

In [ ]:
section("3) Treasury - Daily Cash Forecast")

rows = read_csv("cash_forecast.csv")
running_cash = 0.0

for i, row in enumerate(rows):
    starting_cash = float(row["Starting Cash"])
    inflows = float(row["Inflows"])
    payroll = float(row["Payroll"])
    rent = float(row["Rent"])
    vendor_payments = float(row["Vendor Payments"])
    taxes = float(row["Taxes"])

    outflows = payroll + rent + vendor_payments + taxes
    net_cash_flow = inflows - outflows

    if i == 0:
        running_cash = starting_cash + net_cash_flow
    else:
        running_cash += net_cash_flow

    print(
        f"{row['Date']} | Net Cash Flow: {net_cash_flow:,.2f} | "
        f"Ending Cash: {running_cash:,.2f}"
    )
    if running_cash < 100000:
        print("  LOW CASH WARNING")


7) Treasury - Daily Cash Forecast
2026-02-01 | Net Cash Flow: 40,000.00 | Ending Cash: 540,000.00
2026-02-02 | Net Cash Flow: 90,000.00 | Ending Cash: 630,000.00
2026-02-03 | Net Cash Flow: -50,000.00 | Ending Cash: 580,000.00
2026-02-04 | Net Cash Flow: 45,000.00 | Ending Cash: 625,000.00
2026-02-05 | Net Cash Flow: 35,000.00 | Ending Cash: 660,000.00
2026-02-06 | Net Cash Flow: 37,000.00 | Ending Cash: 697,000.00
